# ⏱️ PyQt6 – Lekce 12: Timery a animace

`QTimer` spouští funkci opakovaně v pravidelných intervalech **bez blokování GUI**.
Použití: hodiny, odpočítávání, animace, načítání dat na pozadí, aktualizace grafů.

> ⚠️ Nikdy nepoužívejte `time.sleep()` v GUI vlákně – zmrazí okno!
> Vždy místo toho použijte `QTimer`.

---

## 📦 Klíčové metody QTimer

| Metoda | Co dělá |
|---|---|
| `QTimer(self)` | Vytvoří timer |
| `.timeout.connect(funkce)` | Signál – zavolá funkci po každém tiknutí |
| `.start(ms)` | Spustí timer s intervalem v milisekundách |
| `.stop()` | Zastaví timer |
| `.isActive()` | Vrátí True, pokud timer běží |
| `.setSingleShot(True)` | Timer se spustí jen jednou (ne opakovaně) |
| `QTimer.singleShot(ms, funkce)` | Zavolá funkci jednou po prodlevě |

---

## 🟢 Ukázka 1 – Digitální hodiny (aktualizace každou sekundu)

In [1]:
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget, QVBoxLayout, QLabel
from PyQt6.QtCore import QTimer, QDateTime
from PyQt6.QtGui import QFont

class Hodiny(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Digitální hodiny")
        self.resize(350, 120)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        self.cas_label = QLabel()
        self.cas_label.setFont(QFont("Courier New", 36, QFont.Weight.Bold))
        self.cas_label.setStyleSheet("color: #00ff88; background: #1a1a2e; padding: 10px; border-radius: 8px;")
        layout.addWidget(self.cas_label)

        # Timer: každou sekundu zavolá aktualizuj()
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.aktualizuj)
        self.timer.start(1000)   # 1000 ms = 1 sekunda

        self.aktualizuj()   # Zobrazíme hned při spuštění (ne až za 1s)

    def aktualizuj(self):
        ted = QDateTime.currentDateTime()
        self.cas_label.setText(ted.toString("HH:mm:ss"))
        self.statusBar().showMessage(ted.toString("dddd d. MMMM yyyy"))


app = QApplication.instance() or QApplication([])
okno = Hodiny()
okno.show()
app.exec()

0

---

## 🟢 Ukázka 2 – Odpočítávání (countdown)

Timer běží každou sekundu, snižujeme čítač a po dosažení nuly ho zastavíme.

In [2]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QLabel, QPushButton, QSpinBox
)
from PyQt6.QtCore import QTimer
from PyQt6.QtGui import QFont

class Odpocitavani(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Odpočítávání")
        self.resize(320, 180)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        # Zobrazení zbývajícího času
        self.displej = QLabel("--")
        self.displej.setFont(QFont("Courier New", 48, QFont.Weight.Bold))
        self.displej.setStyleSheet("color: #ff6b6b; padding: 5px;")
        layout.addWidget(self.displej)

        # Nastavení a ovládání
        ovladani = QHBoxLayout()
        self.spinbox = QSpinBox()
        self.spinbox.setRange(1, 600)
        self.spinbox.setValue(10)
        self.spinbox.setSuffix(" s")

        self.btn_start = QPushButton("▶ Start")
        self.btn_stop  = QPushButton("⏹ Stop")
        self.btn_start.clicked.connect(self.start)
        self.btn_stop.clicked.connect(self.stop)
        self.btn_stop.setEnabled(False)

        ovladani.addWidget(QLabel("Čas:"))
        ovladani.addWidget(self.spinbox)
        ovladani.addWidget(self.btn_start)
        ovladani.addWidget(self.btn_stop)
        layout.addLayout(ovladani)

        self.timer = QTimer(self)
        self.timer.timeout.connect(self.tik)
        self.zbyvajici = 0

    def start(self):
        self.zbyvajici = self.spinbox.value()
        self.displej.setStyleSheet("color: #ff6b6b; padding: 5px;")
        self.aktualizuj_displej()
        self.timer.start(1000)
        self.btn_start.setEnabled(False)
        self.btn_stop.setEnabled(True)
        self.statusBar().showMessage("Odpočítávám…")

    def stop(self):
        self.timer.stop()
        self.btn_start.setEnabled(True)
        self.btn_stop.setEnabled(False)
        self.statusBar().showMessage("Zastaveno")

    def tik(self):
        self.zbyvajici -= 1
        self.aktualizuj_displej()
        if self.zbyvajici <= 0:
            self.timer.stop()
            self.displej.setStyleSheet("color: #00ff88; padding: 5px;")
            self.displej.setText("✅")
            self.btn_start.setEnabled(True)
            self.btn_stop.setEnabled(False)
            self.statusBar().showMessage("Hotovo! 🎉")

    def aktualizuj_displej(self):
        minuty = self.zbyvajici // 60
        sekundy = self.zbyvajici % 60
        self.displej.setText(f"{minuty:02d}:{sekundy:02d}")


app = QApplication.instance() or QApplication([])
okno = Odpocitavani()
okno.show()
app.exec()

0

---

## 🟡 Ukázka 3 – Animace pohybujícího se prvku (QLabel)

Pohybujeme prvkem po obrazovce pomocí `move()`. Timer zavolá pohyb každých 50 ms (= ~20 fps).

In [3]:
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget, QLabel, QPushButton, QHBoxLayout
from PyQt6.QtCore import QTimer
from PyQt6.QtGui import QFont

class AnimaceApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Animace pohybu")
        self.setFixedSize(600, 300)

        central = QWidget(self)
        self.setCentralWidget(central)
        central.setStyleSheet("background: #1a1a2e;")

        # Pohybující se míček
        self.miacek = QLabel("⚽", central)
        self.miacek.setFont(QFont("", 32))
        self.miacek.move(0, 130)

        # Tlačítka
        btn_widget = QWidget(central)
        btn_widget.move(230, 250)
        btn_layout = QHBoxLayout(btn_widget)

        btn_start = QPushButton("▶ Start")
        btn_stop  = QPushButton("⏹ Stop")
        btn_start.clicked.connect(self.start)
        btn_stop.clicked.connect(self.stop)
        btn_layout.addWidget(btn_start)
        btn_layout.addWidget(btn_stop)

        self.x = 0
        self.smer = 1    # 1 = doprava, -1 = doleva
        self.rychlost = 5

        self.timer = QTimer(self)
        self.timer.timeout.connect(self.pohni)

    def start(self):
        self.timer.start(50)   # 50 ms = 20 fps

    def stop(self):
        self.timer.stop()

    def pohni(self):
        sirka_okna = self.centralWidget().width()
        self.x += self.rychlost * self.smer

        # Odraz od stěn
        if self.x >= sirka_okna - 50:
            self.smer = -1
        elif self.x <= 0:
            self.smer = 1

        self.miacek.move(self.x, self.miacek.y())


app = QApplication.instance() or QApplication([])
okno = AnimaceApp()
okno.show()
app.exec()

0

---

## 🔴 Ukázka 4 – QTimer.singleShot (jednorázové spuštění)

Někdy chceme provést akci **jednou po určité prodlevě** – třeba schovat zprávu
po 3 sekundách nebo načíst data s malým zpožděním.

In [5]:
from PyQt6.QtWidgets import QApplication, QMainWindow, QWidget, QVBoxLayout, QLabel, QPushButton
from PyQt6.QtCore import QTimer

class ZpravaApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Zpráva s timeoutem")
        self.resize(350, 120)

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        self.zprava = QLabel("")
        self.zprava.setStyleSheet("color: green; font-size: 14px; font-weight: bold;")
        btn = QPushButton("Uložit (simulace)")
        btn.clicked.connect(self.ulozit)

        layout.addWidget(btn)
        layout.addWidget(self.zprava)

    def ulozit(self):
        self.zprava.setText("✅ Uloženo!")
        # Schová zprávu za 3 sekundy – pouze jednou
        QTimer.singleShot(3000, self.skryj_zpravu)

    def skryj_zpravu(self):
        self.zprava.setText("")


app = QApplication.instance() or QApplication([])
okno = ZpravaApp()
okno.show()
app.exec()

0

---

## 📋 Shrnutí lekce

| Situace | Řešení |
|---|---|
| Opakovaná akce každých N ms | `QTimer` + `.start(N)` |
| Jednorázová akce po N ms | `QTimer.singleShot(N, funkce)` |
| Zastavit timer | `.stop()` |
| Zjistit, zda běží | `.isActive()` |
| Blokující čekání (**ŠPATNĚ**) | ~~`time.sleep()`~~ |

## ✏️ Úkoly

1. Napište aplikaci **Stopky** s tlačítky Start, Stop a Reset. Zobrazujte čas v sekundách i desetinách sekundy (timer každých 100 ms).
2. Rozšiřte animaci míčku na **2D pohyb** – míček se odráží od všech čtyř stran okna.
3. Vytvořte **"Loading" screen** – po spuštění aplikace zobrazíte progress bar, který se za 3 sekundy naplní (timer každých 300 ms přidá 10 %) a poté zmizí.